# K-Nearest Neighbors (KNN) — T-Shirt Size Demo

KNN is one of the simplest and most intuitive supervised learning algorithms. It is a **lazy learner** — it stores the entire training set and defers computation until prediction time.

## Key Concepts

**How KNN Works**
1. Given a new sample, compute its distance to every training sample.
2. Select the K closest neighbours.
3. Assign the majority class among those K neighbours (classification).

**Distance Metrics**
- **Euclidean**: `d = sqrt(sum((x_i - y_i)^2))` — straight-line distance (most common)
- **Manhattan**: `d = sum(|x_i - y_i|)` — sum of absolute differences along each axis
- **Minkowski**: generalisation of both (`p=2` -> Euclidean, `p=1` -> Manhattan)

**Choosing K**
- Small K (e.g. 1): highly flexible boundary, prone to overfitting / noise sensitivity
- Large K: smoother boundary, more robust, but may under-fit
- Common approach: cross-validate across a range of K values

**Voting Schemes**
- **Uniform**: all K neighbours get an equal vote
- **Distance-weighted**: closer neighbours have more influence (weight = 1/distance)

**Important**: KNN is **distance-based**, so features must be on the same scale — `StandardScaler` is essential.

---

In this notebook we will:
1. Load the T-Shirt size dataset (Height, Weight -> Size M or L)
2. Explore and visualise the 2-D feature space
3. Train KNN with different K values and distance metrics
4. Visualise decision boundaries directly (no PCA needed — only 2 features!)
5. Tune hyperparameters and evaluate the best model
6. Make custom predictions for new inputs

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from data_loader import (
    load_tshirt_data,
    split_features_target,
    encode_target,
    get_feature_names,
)
from model import (
    scale_features,
    train_knn,
    evaluate_model,
    predict,
    find_best_k,
    tune_hyperparameters,
)
from visualization import (
    plot_class_distribution,
    plot_scatter,
    plot_decision_boundary,
    plot_k_vs_accuracy,
    plot_confusion_matrix,
    plot_metric_comparison,
)

sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42

%matplotlib inline

## 2. Load & Explore Data

The T-Shirt Size dataset has just 18 samples, 2 numeric features (Height and Weight), and a binary target (M or L). Its simplicity makes it ideal for visualising KNN concepts.

In [ ]:
df = load_tshirt_data()
X, y = split_features_target(df)

print(f"Dataset shape: {df.shape}")
print(f"Features     : {get_feature_names()}")
print(f"Target values: {y.unique().tolist()}")
print()
df

In [ ]:
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
plot_class_distribution(y)
plt.show()

In [ ]:
plot_scatter(X, y, get_feature_names())
plt.show()

## 4. Data Preprocessing

We encode the target labels (M -> 0, L -> 1), split into train/test, and apply StandardScaler so that Height and Weight contribute equally to distance calculations.

In [ ]:
y_enc, encoder = encode_target(y)
print(f"Label mapping: {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=RANDOM_STATE
)
X_train_sc, X_test_sc, scaler = scale_features(X_train, X_test)

print(f"\nTraining set : {X_train_sc.shape[0]} samples")
print(f"Test set     : {X_test_sc.shape[0]} samples")

## 5. KNN with K=3

In [ ]:
knn_3 = train_knn(X_train_sc, y_train, n_neighbors=3)
results_3 = evaluate_model(knn_3, X_test_sc, y_test)

print(f"KNN (K=3) — Test Accuracy: {results_3['accuracy']:.4f}")
print(f"\n{results_3['report']}")

## 6. KNN with K=5

In [ ]:
knn_5 = train_knn(X_train_sc, y_train, n_neighbors=5)
results_5 = evaluate_model(knn_5, X_test_sc, y_test)

print(f"KNN (K=5) — Test Accuracy: {results_5['accuracy']:.4f}")
print(f"\n{results_5['report']}")

comparison = pd.DataFrame({
    "K": [3, 5],
    "Accuracy": [results_3["accuracy"], results_5["accuracy"]],
})
comparison

## 7. Effect of K

We cross-validate K = 1 through 13 (limited by the small training set size) and plot accuracy vs K to find the sweet spot.

In [ ]:
max_k = min(13, len(y_train))
k_scores = find_best_k(X_train_sc, y_train,
                        k_range=range(1, max_k + 1),
                        cv=min(len(y_train), 5))

best_k = max(k_scores, key=k_scores.get)
print(f"Best K = {best_k}  (Mean CV Accuracy = {k_scores[best_k]:.4f})")
print("\nAll scores:")
for k, acc in k_scores.items():
    marker = " <-- best" if k == best_k else ""
    print(f"  K={k:2d}: {acc:.4f}{marker}")

In [ ]:
plot_k_vs_accuracy(k_scores)
plt.show()

## 8. Decision Boundary Visualization

With only 2 features we can plot decision boundaries directly on the scaled feature space — no dimensionality reduction needed. We compare K=1, 3, 5, and 9 to show how increasing K smooths the boundary.

In [ ]:
X_all_sc = scaler.transform(X)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, k in zip(axes.flatten(), [1, 3, 5, 9]):
    knn_viz = train_knn(X_all_sc, y_enc, n_neighbors=k)
    plot_decision_boundary(knn_viz, X_all_sc, y_enc,
                           title=f"K = {k}", ax=ax)
plt.suptitle("Effect of K on Decision Boundary", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 9. Distance Metrics Comparison

We compare Euclidean, Manhattan, and Minkowski (p=2, equivalent to Euclidean) distances with the best K found above.

In [ ]:
metric_results = {}
for metric in ["euclidean", "manhattan", "minkowski"]:
    m = train_knn(X_train_sc, y_train, n_neighbors=best_k, metric=metric)
    r = evaluate_model(m, X_test_sc, y_test)
    metric_results[metric.title()] = r["accuracy"]
    print(f"{metric:>12s} — Accuracy: {r['accuracy']:.4f}")

plot_metric_comparison(metric_results)
plt.title("Distance Metric Comparison")
plt.show()

## 10. Weighted vs Uniform Voting

With **uniform** voting every neighbour gets one vote. With **distance** weighting closer neighbours contribute more heavily.

In [ ]:
weight_results = {}
for w in ["uniform", "distance"]:
    m = train_knn(X_train_sc, y_train, n_neighbors=best_k, weights=w)
    r = evaluate_model(m, X_test_sc, y_test)
    weight_results[w.title()] = r["accuracy"]
    print(f"{w:>10s} — Accuracy: {r['accuracy']:.4f}")

plot_metric_comparison(weight_results)
plt.title("Uniform vs Distance-Weighted Voting")
plt.show()

## 11. Hyperparameter Tuning with GridSearchCV

We search over `n_neighbors`, `metric`, and `weights` simultaneously using 5-fold cross-validation (or fewer folds if the training set is very small).

In [ ]:
search = tune_hyperparameters(X_train_sc, y_train,
                              cv=min(len(y_train), 5))

print(f"Best parameters : {search.best_params_}")
print(f"Best CV accuracy: {search.best_score_:.4f}")

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
top_10 = cv_results.nsmallest(10, "rank_test_score")[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
]
top_10

## 12. Final Evaluation

Evaluate the best model from GridSearchCV on the held-out test set.

In [ ]:
best_model = search.best_estimator_
best_results = evaluate_model(best_model, X_test_sc, y_test)

print(f"Best KNN — Test Accuracy: {best_results['accuracy']:.4f}")
print(f"Parameters: {search.best_params_}")
print(f"\nClassification Report:\n{best_results['report']}")

In [ ]:
plot_confusion_matrix(best_results["confusion_matrix"],
                      class_names=list(encoder.classes_))
plt.show()

In [ ]:
best_k_final = search.best_params_["n_neighbors"]
knn_final_viz = train_knn(X_all_sc, y_enc, n_neighbors=best_k_final,
                          metric=search.best_params_["metric"],
                          weights=search.best_params_["weights"])
plot_decision_boundary(knn_final_viz, X_all_sc, y_enc,
                       title=f"Best Model Decision Boundary (K={best_k_final})")
plt.show()

## 13. Custom Prediction

Let's predict the T-shirt size for a new person. We scale the input using the same scaler fitted on training data, then use the best model to predict.

In [ ]:
new_samples = pd.DataFrame({
    "Height (in cms)": [162, 172, 155],
    "Weight (in kgs)": [60, 67, 57],
})

new_scaled = scaler.transform(new_samples)
predictions = predict(best_model, new_scaled)
predicted_labels = encoder.inverse_transform(predictions)

result_df = new_samples.copy()
result_df["Predicted Size"] = predicted_labels
print("Custom Predictions:")
result_df

## 14. Conclusion

**Key Takeaways**

1. **KNN is simple but powerful** — no training phase; the algorithm memorises the data and classifies new points by majority vote among the K closest neighbours.

2. **Feature scaling is essential** — Height ranges from 158-170 while Weight ranges from 58-68. Without scaling, Height would dominate distance calculations. `StandardScaler` equalises their contributions.

3. **K controls the bias-variance trade-off** — K=1 perfectly fits training data but is noisy; larger K produces smoother boundaries but may under-fit. The elbow plot helps identify the sweet spot.

4. **Decision boundaries are intuitive** — with 2 features we can directly visualise how increasing K smooths the decision surface from jagged (K=1) to a clean separating region.

5. **Distance metric choice** — Euclidean and Manhattan performed comparably here. For higher-dimensional data the choice can matter more.

6. **Small datasets benefit from LOOCV** — with only 18 samples, leave-one-out or small-fold cross-validation gives more reliable estimates than a single train/test split.

**Next Steps**
- Try on larger, higher-dimensional datasets where the curse of dimensionality becomes relevant
- Compare KNN to other classifiers (Decision Trees, SVM, Logistic Regression)
- Experiment with feature engineering (e.g. BMI = Weight / Height^2)